In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

In [4]:
import joblib
import pandas as pd
from sklearn.metrics import silhouette_score

features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
            'DiabetesPedigreeFunction', 'Age']

df = pd.read_csv("../data/processed/diabetes_clustered.csv")
kmeans_final = joblib.load("../models/kmeans_model.joblib")
scaler_final = joblib.load("../models/cluster_scaler.joblib")

X_scaled = scaler_final.transform(df[features])
labels = kmeans_final.predict(X_scaled)
sil = silhouette_score(X_scaled, labels)

print(kmeans_final.n_clusters, round(kmeans_final.inertia_), round(sil, 3))

2 4071 0.232


c:\Users\yassi\Desktop\diabetes-risk-mlops\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but KMeans was fitted with feature names
  warnings.warn(


In [7]:

import mlflow


mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("diabetes-clustering")


with mlflow.start_run(run_name="kmeans-k2"):
    mlflow.log_param("k",kmeans_final.n_clusters)
    mlflow.log_param("features",", ".join(features))
    
    for name,mean,std in zip(features, scaler_final.mean_, scaler_final.scale_):
        mlflow.log_param(f"scaler_mean_{name}",round(mean,4))
        mlflow.log_param(f"scaler_std_{name}",round(std,4))
        
        
    mlflow.log_metric("inertia",kmeans_final.inertia_)
    mlflow.log_metric("silhouette_score", sil)
    
    mlflow.log_artifact("../models/kmeans_model.joblib")
    mlflow.log_artifact("../models/cluster_scaler.joblib")

print("Clustering run saved")

Clustering run saved


In [8]:
from src.tracking import log_clustering_run

log_clustering_run(
    kmeans_final, scaler_final, features, sil,
    model_path="../models/kmeans_model.joblib",
    scaler_path="../models/cluster_scaler.joblib",
)

In [11]:
from sklearn.model_selection import train_test_split
from src.tracking import log_classification_run

X = df[features]
y = df["Cluster"]
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline = joblib.load("../models/diabetes_risk_pipeline.joblib")

log_classification_run(
    pipeline, X_test, y_test,
    params={"model": "LogisticRegression", "C": 10, "solver": "lbfgs"},
    features=features,
    run_name="logistic-regression-tuned",
    model_path="../models/diabetes_risk_pipeline.joblib",
)